# Autocrop — Automatic Square Cropping with Saliency Map

## Automatic Square Cropping

This notebook automatically crops images into squares by finding the most visually important region using a **saliency map** computed from VGG19.

```
Input image → Resize → Saliency map (VGG19 gradient) → Find best crop window → Square crop
```

### How it works

1. **Resize** — Scale the image so the shorter side equals `size` pixels
2. **Saliency map** — Run VGG19 and compute per-pixel gradient magnitude; brighter = more salient
3. **Best window** — Slide a `size`-wide window along the longer axis and pick the position with the highest total saliency
4. **Crop & save** — Crop the image to that window and save

### Output folders

| Folder | Contents |
|--------|----------|
| `outputs/` | Cropped square images |
| `outputs_saliency/` | Saliency maps visualized with the jet colormap |

### Key options

| Argument | Default | Description |
|----------|---------|-------------|
| `--inputs` | `inputs` | Folder containing input images |
| `--outputs` | `outputs` | Folder to save cropped images |
| `--outputs_saliency` | `outputs_saliency` | Folder to save saliency map images |
| `--size` | `256` | Output square size in pixels |

📄 [torchvision.models docs](https://pytorch.org/vision/stable/models.html)

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/Practical-ML/autocrop"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

Mounted at /content/drive
/content/drive/MyDrive/Practical-ML/autocrop


In [2]:
# Download input images from GitHub
import os

# Download only the inputs folder from GitHub (shallow sparse checkout)
if not os.path.exists(f'{PROJECT_PATH}/inputs'):
    !git clone --depth=1 --filter=blob:none --sparse https://github.com/mastnk/practical-ml /tmp/practical-ml -q
    !cd /tmp/practical-ml && git sparse-checkout set autocrop/inputs
    !cp -r /tmp/practical-ml/autocrop/inputs "{PROJECT_PATH}/inputs"
    print('Images downloaded.')
else:
    print('Images already exist, skipping.')

%cd "{PROJECT_PATH}"
!ls inputs/

remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 3.68 KiB | 3.68 MiB/s, done.
remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 455.24 KiB | 1.76 MiB/s, done.
Images downloaded.
/content/drive/MyDrive/Practical-ML/autocrop
AKANE278123825_TP_V.jpg  SAYA-PAKU5465_TP_V.jpg


In [3]:
# Save this cell as a Python file (Execute after editing)
%%writefile autosquarecrop.py
import argparse

from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt

import torch
import torchvision
import torchvision.transforms as transforms

import os
import glob
import numpy as np

def load_images( dir, exts=["jpg", "png", "bmp"] ):
    exts = [e.lower() for e in exts] + [e.upper() for e in exts]

    filenames = []
    for ext in exts:
        filenames += sorted( glob.glob( dir+"/*.{}".format(ext) ) )

    images = []
    for filename in filenames:
        images += [Image.open(filename).convert('RGB')]
    return filenames, images

def saliencymap( model, image ):
    model.eval()
    image_tensor = transforms.ToTensor()( image )
    image_tensor = torch.unsqueeze(image_tensor, 0)
    input_tensor = image_tensor.requires_grad_()
    output_tensor = model(input_tensor)
    grads = torch.autograd.grad(outputs=output_tensor, inputs=input_tensor,
                                grad_outputs=torch.ones(output_tensor.size()),
                                create_graph=True, retain_graph=True, only_inputs=True)[0]
    grads = grads.detach()
    grads = grads[0]
    grads = grads.square().sum(axis=0).sqrt()
    grads = grads / grads.max()
    return grads.numpy()

def save_with_colormap( filename, array, colormap="jet" ):
    cmap = plt.get_cmap(colormap)
    array = cmap(array, bytes=True)
    im = Image.fromarray(array)
    im.save(filename)

def main( inputs, outputs, outputs_saliency, size ):
    # Load a pre-trained model
    model = None
    try:
        model = torchvision.models.vgg19(weights="IMAGENET1K_V1", progress=False)
    except:
        pass

    if( model is None ):
        model = torchvision.models.vgg19(pretrained=True, progress=False)


    os.makedirs( outputs, exist_ok=True)
    os.makedirs( outputs_saliency, exist_ok=True)

    filenames, images = load_images( inputs )

    for filename, image in zip(filenames,images):
        title = os.path.splitext( os.path.split( filename )[1] )[0]
        print( title )
        s = image.size
        if( s[0] >= s[1] ): #width>=height
            a = float(size)/float(s[1])
            s = [int(s[0]*a),size]
        else: #width<height
            a = float(size)/float(s[0])
            s = [size, int(s[1]*a)]

        image = image.resize(s, Image.LANCZOS)

        sal = saliencymap( model, image )
        save_with_colormap( os.path.join( outputs_saliency, title+".png" ), sal )

        if( s[0] < s[1] ): #width<height
            image = image.transpose(Image.Transpose.ROTATE_90)
            sal = sal.transpose(1,0)

        x = sal.mean( axis=0 )
        x = np.convolve(x, np.ones(size), mode='valid')
        p = x.argmax()

        left = p
        right = p+size
        top = 0
        bottom=size
        image = image.crop((left, top, right, bottom))

        if( s[0] < s[1] ): #width<height
            image = image.transpose(Image.Transpose.ROTATE_270)
            sal = sal.transpose(1,0)

        image.save( os.path.join( outputs, title+".png" ) )


###
if( __name__ == "__main__" ):
    parser = argparse.ArgumentParser(description="Automatic image cropping")

    parser.add_argument("--inputs", default="inputs")
    parser.add_argument("--outputs", default="outputs")
    parser.add_argument("--outputs_saliency", default="outputs_saliency")
    parser.add_argument("--size", type=int, default=256)

    cfg = vars( parser.parse_args() )
    main( **cfg )


Writing autosquarecrop.py


In [4]:
# Execute a Python file
%cd "{PROJECT_PATH}"
!python autosquarecrop.py

/content/drive/MyDrive/Practical-ML/autocrop
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
AKANE278123825_TP_V
SAYA-PAKU5465_TP_V


💾 **Don't forget to save this notebook!**

To keep your work, go to File → Save a copy in Drive before closing.

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)